In [7]:
import pandas as pd
from pymongo import MongoClient
from math import radians, cos, sin, asin, sqrt

def haversine(lon1, lat1, lon2, lat2):
    """ Berechnet die Entfernung zwischen zwei Punkten (Dezimalgrad) """
    lon1, lat1, lon2, lat2 = map(radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1 
    dlat = lat2 - lat1 
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * asin(sqrt(a)) 
    r = 6371 # Erdradius in km
    return c * r

def berechne_autonomie_index():
    client = MongoClient("mongodb://localhost:27017/")
    db = client.Espana
    collection = db.ciudad
    
    print("--- STRATEGISCHE ANALYSE V2: AUTONOMES WOHNEN ---\n")
    
    staedte = list(collection.find())
    results = []

    for s in staedte:
        score = 0
        merkmale = []
        
        # 1. Bahn-Check (Jetzt mit Entfernungs-Logik)
        estaciones = s.get('Estacion', [])
        if isinstance(estaciones, list) and len(estaciones) > 0:
            # Wir nehmen die erste Station
            est_coords = estaciones[0].get('Coordinates', {}).get('coordinates')
            city_coords = s.get('Coordinates', {}).get('coordinates')
            
            if est_coords and city_coords:
                dist = haversine(city_coords[0], city_coords[1], est_coords[0], est_coords[1])
                if dist <= 2:
                    score += 40
                    merkmale.append(f"Bahn (<2km)")
                elif dist <= 5:
                    score += 20
                    merkmale.append(f"Bahn (E-Bike Distanz)")
                else:
                    score += 5
                    merkmale.append("Bahn (weit weg)")
        elif isinstance(estaciones, str) and estaciones != 'No': # Fallback für altes Format
            score += 20
            merkmale.append("Bahn (Check Distanz)")

        # 2. Medizinische Versorgung (Hospital-Check)
        hospital = s.get('Hospital')
        if hospital and hospital != 'No':
            score += 30
            merkmale.append("Hospital")

        # 3. Einkaufsmöglichkeiten (Supermarkt/Markthalle)
        supers = s.get('Supermercado', [])
        if (isinstance(supers, list) and len(supers) > 0) or (isinstance(s.get('Tienda'), str) and len(s.get('Tienda', '')) > 5):
            score += 20
            merkmale.append("Shopping")
            
        # 4. Venen- & Fahrrad-Faktor (Höhe < 100m)
        altitud = s.get('Altitud', 1000)
        if isinstance(altitud, (int, float)) and altitud < 100:
            score += 10
            merkmale.append("Flach")

        # Filter: Nur Orte mit einer gewissen Basis an Infrastruktur
        if score >= 40:
            results.append({
                "Stadt": s.get('Ciudad'),
                "Provinz": s.get('Provincia'),
                "Score": score,
                "Infrastruktur": ", ".join(merkmale),
                "Höhe": f"{altitud}m",
                "Dichte": s.get('Densidad', 0)
            })

    df = pd.DataFrame(results)
    if not df.empty:
        df = df.sort_values(by="Score", ascending=False)
        print(df.to_string(index=False))
        print(f"\nAnalyse abgeschlossen. {len(df)} Städte erfüllen die Kriterien.")
    else:
        print("Keine Städte gefunden, die den Kriterien entsprechen.")

berechne_autonomie_index()

--- STRATEGISCHE ANALYSE V2: AUTONOMES WOHNEN ---

                    Stadt                Provinz  Score                                   Infrastruktur   Höhe  Dichte
                  Granada                Granada     90                 Bahn (<2km), Hospital, Shopping   684m 2658.20
                  Llerena                Badajoz     90                 Bahn (<2km), Hospital, Shopping   510m   98.74
               Villarreal              Castellón     80 Bahn (Check Distanz), Hospital, Shopping, Flach    42m  913.17
                La Coruña              La Coruña     80 Bahn (Check Distanz), Hospital, Shopping, Flach    21m 6452.52
               La Guardia             Pontevedra     80 Bahn (Check Distanz), Hospital, Shopping, Flach     0m  490.29
                 Alicante               Alicante     80 Bahn (Check Distanz), Hospital, Shopping, Flach     5m 1639.53
                   Málaga                 Málaga     80 Bahn (Check Distanz), Hospital, Shopping, Flach     8m 1440.

In [5]:
import pandas as pd
from pymongo import MongoClient
from math import radians, cos, sin, asin, sqrt

def haversine(lon1, lat1, lon2, lat2):
    """ Berechnet die Distanz in km zwischen zwei Koordinaten-Paaren """
    lon1, lat1, lon2, lat2 = map(radians, [lon1, lat1, lon2, lat2])
    dlon, dlat = lon2 - lon1, lat2 - lat1 
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    return 2 * asin(sqrt(a)) * 6371

def berechne_autonomie_index():
    client = MongoClient("mongodb://localhost:27017/")
    db = client.Laender
    collection = db.Espana
    
    print("--- STRATEGISCHE ANALYSE V2.1: KLIMA, MOBILITÄT & INFRASTRUKTUR ---\n")
    
    staedte = list(collection.find())
    results = []

    for s in staedte:
        score = 0
        merkmale = []
        
        # 1. MOBILITÄT: Bahn-Erreichbarkeit (Karlsruher Maßstab)
        estaciones = s.get('Estacion', [])
        if isinstance(estaciones, list) and len(estaciones) > 0:
            est_coords = estaciones[0].get('Coordinates', {}).get('coordinates')
            city_coords = s.get('Coordinates', {}).get('coordinates')
            if est_coords and city_coords:
                dist = haversine(city_coords[0], city_coords[1], est_coords[0], est_coords[1])
                if dist <= 2:
                    score += 40
                    merkmale.append("Bahn (Zentrum)")
                elif dist <= 5:
                    score += 20
                    merkmale.append("Bahn (E-Bike)")
                else:
                    score -= 10 # Abzug für "Haren-Effekt" (weit außerhalb)
                    merkmale.append("Bahn (Abgelegen)")
        
        # 2. KLIMA: Hitze-Check (Venen-Schutz)
        temp_high = s.get('Temperature high', {})
        if temp_high:
            juli_temp = temp_high.get('Jul', 0)
            aug_temp = temp_high.get('Aug', 0)
            max_sommer = max(juli_temp, aug_temp)
            
            if max_sommer > 35:
                score -= 30 # "Bratpfannen"-Abzug (Lora del Río Syndrom)
                merkmale.append("Extreme Hitze!")
            elif max_sommer > 32:
                score -= 10
                merkmale.append("Heiß")
            else:
                score += 15
                merkmale.append("Mildes Klima")

        # 3. VERSORGUNG: Hospital & Shopping
        if s.get('Hospital') and s.get('Hospital') != 'No':
            score += 25
            merkmale.append("Hospital")
            
        supers = s.get('Supermercado', [])
        if (isinstance(supers, list) and len(supers) > 0):
            score += 15
            merkmale.append("Supermarkt")

        # 4. DICHTE-CHECK: (Vermeidung von "Essen-Gefühl")
        dichte = s.get('Densidad', 0)
        if 50 <= dichte <= 1000: # Ideale Dichte für Ruhe + Infrastruktur
            score += 10
            merkmale.append("Gute Dichte")

        if score >= 30:
            results.append({
                "Stadt": s.get('Ciudad'),
                "Provinz": s.get('Provincia'),
                "Score": score,
                "Juli-Temp": f"{temp_high.get('Jul')}°C" if temp_high else "N/A",
                "Merkmale": ", ".join(merkmale),
                "Dichte": f"{dichte:.1f}"
            })

    df = pd.DataFrame(results).sort_values(by="Score", ascending=False)
    print(df.to_string(index=False))

berechne_autonomie_index()

--- STRATEGISCHE ANALYSE V2.1: KLIMA, MOBILITÄT & INFRASTRUKTUR ---



KeyError: 'Score'